**Go** è un linguaggio progettato per facilitare la costruzione di software semplice, affidabile e scalabile.  
Queste caratteristiche sono particolarmente importanti nei sistemi distribuiti, dove tocca gestire tante attività contemporaneamente e programmi/comunicazioni di rete che devono funzionare anche al crescere del carico. 

Go nasce nel 2007 in Google, i suoi obiettivi principali sono:
- **efficienza del software** durante l'esecuzione
- **semplicità di sviluppo**

Per quanto riguarda il primo obiettivo Go è **linguaggio compilato** e **tipizzato staticamente** (i.e. i tipi delle variabili sono verificati prima dell'esecuzione); per il secondo sintassi semplice e nell'esperienza d'uso che sembri più simile a un linguaggio interpretato.

Due riferimenti importanti per Go sono **C** e **CSP**. 

Dal C derivano elementi come la sintassi delle espressioni, i costrutti per il controllo del flusso come `if`, `for`, `switch`, tipizzazione base e *puntatori*, passaggio di parametri per valore (i.e. la funzione riceve copia del valore passato come argomento) e l'attenzione all'efficienza in fase di esecuzione.  

Il CSP (Communicating Sequential Processes) è invece un modello per descrivere sistemi costituiti da attività sequenziali, che procedono in modo concorrente e comunicano tra loro.  
Di sto CSP bisogna ricordare quest'idea chiave: **all'interno di ogni attività le istruzioni seguono un ordine, ma più attività possono avanzare concorrentemente e coordinarsi tramite comunicazione**; da qui deriva la base delle **goroutine** e dei **canali** in Go.

Go viene introdotto per tre ragioni: prima cosa la **distribuzione del programma** in quanto Go può produrre un singolo eseguibile binario che nelle configurazioni adatte contiene tutto il necessario per l'esecuzione --> facilita il trasferimento e l'avvio del software sulle macchine destinazione.  
La seconda è il **supporto alla programmazione di rete**, in quanto Go dispone di librerie necessarie ad utilizzare meccanismi come RPC e gRPC.  
Il terzo elemento è il **modello di concorrenza**, basato su **goroutine** (attività concorrenti leggere), **channel** (canali attraverso cui le attività comunicano) e **select** (costrutto che permette di attendere su più canali).

È il runtime di Go ad occuparsi della gestione delle goroutine e di distribuirne l'esecuzione sui thread SO --> *una goroutine non corrisponde necessariamente a un thread dedicato*.

Go offre anche **struct**, **metodi** e **interfacce** ma non nel senso classico OOP, quanto più come strumenti per organizzare il codice e definire comportamenti (i.e. non c'è ereditarietà, ma solo composizione).

Infine Go ha anche un meccanismo di **garbage collection**.

#### Esempio
Vediamo un esempio per iniziare a capire la sintassi di Go:

<img src="imgs/go1.png" alt="go1" width="400"/>

Qui il programma avvia tre attività concorrenti e raccoglie i msg attraverso un canale (stiamo simulando tre risposte da server).  
Con `ch := make(chan string, 3)` si ha `chan string` che indica un canale che trasporta valori di tipo stringa, `make` che crea e inizializza il canale, `3` che indica che il canale contiene un buffer che può contenere 3 valori e `:=` dichiara una variabile e la inizializza, facendo dedurre al compilatore il tipo.

Il canale è creato prima delle goroutine affinché lo possano usare per comunicare col `main`. 

Il primo `for` esegue tre iterazioni con `i` da 1 a 3 (sintassi uguale a C ma no parentesi tonde); la parola chiave `go` davanti a una funzione dice che deve essere eseguita come goroutine --> il `main` può proseguire senza aspettare che quella chiamata termini.  
In questo caso si usa una funzione anonima `func(id int)`.  
Ciascuna goroutine attende 100 ms con la sleep, costruisce una stringa con `fmt.Sprintf` e la invia al canale con `ch <- msg`.

Qui importante il senso della freccia: `ch <- msg` significa "invia il valore `msg` al canale `ch`", mentre `r := <-ch` significa "ricevi un valore dal canale `ch` e assegnalo a `r`".

Dopo aver avviato le goroutine, il `main` esegue un secondo `for` dove ad ogni iterazione, `select` attende che possa procedere uno dei due casi:
- `case r := <-ch:` se arriva un valore dal canale `ch`, lo assegna a `r` e stampa il messaggio ricevuto
- `case <-time.After(time.Second):` se non arriva alcun valore dal canale entro un secondo, stampa un messaggio di timeout (`time.After` fornisce un canale sul quale arriva una notifica dopo il tempo specificato).

Chiaramente l'ordine delle risposte non è garantito: avviare le tre goroutine in ordine non obbliga a ricevere i msg nello stesso ordine.

#### Esempio Hello World
```go
package main
import "fmt"

func main() {
    fmt.Println("Hello, 世界!")
}
```
`package main` indica che il file fa parte del package `main`, ogni package può contenere più file `.go` della stessa dir, come in Java.  
Vabbé l'import per usare il package `fmt` che offre le funzioni per input e output formattati, e la funzione `main` che stampa l'hello world. 

La presenza di caratteri cinesi evidenzia il supporto a Unicode, e si nota anche l'assenza di ; alla fine delle istruzioni che è opzionale.

#### Compilazione ed esecuzione; packages
Go ha una propria **toolchain**, i.e. ha strumenti integrati per compilare ed eseguire il codice:
- `go run helloworld.go` compila ed esegue il file `helloworld.go`
- `go build helloworld.go` compila il file e produce un eseguibile binario nella stessa dir
- `go install helloworld.go` compila il file e produce un eseguibile binario nella destinazione prevista dalla toolchain

Le prime due sono quelle che ci interessano di più, la distinzione è che con `go run` compilazione e avvio avvengono svolti insieme senza lasciare nella directory il normale eseguibile prodotto da `go build` --> sempre e comunque linguaggio compilato.

Un **package** può contenere dichiarazioni di tipi, funzioni, variabili e costanti. Una regola fondamentale di Go è che **l'iniziale del nome determina se un identificatore è esportato**, i.e. accessibile al'esterno del package: **se l'iniziale è maiuscola l'identificatore è esportato, altrimenti no**.

Da qui il motivo della P in `fmt.Println`: la funzione deve essere esportata dal package `fmt` per poter essere utilizzata dal nostro codice.

es. `math.pi` con `math` package importato produce errore perché tutte le funzioni utilizzabili fuori dal package iniziano sempre con lettera maiuscola!

Si possono raggruppare più import in un blocco:
```go
import (
    "fmt"
    "math"
)

func main() {
    fmt.Printf("now you have %g problems.\n", math.Sqrt(7))
}
```
Differenza tra `Println` e `Printf` (entrambe package `fmt`): la prima stampa il messaggio e va a capo, la seconda stampa il messaggio senza andare a capo e permette di formattare l'output con specificatori di formato come `%d`, `%s`, `%f`, `%g` ecc.. per andare a capo in sto caso esplicita `\n`

Altra cosa importante in Go è che **il tipo segue il nome del parametro**, al contrario di C/C++/Java:

```go
func add(x int, y int) int {
    return x + y
}
```
oppure si può scrivere in forma compatta per più parametri dello stesso tipo:
```go
func add(x, y int) int {
    return x + y
}
```

Altra caratteristica è che **una funzione può restituire più risultati distinti**:
```go
func swap(x, y string) (string, string) {
    return y, x
}
```

In Go inoltre la gestione degli errori è esplicita in quanto molte funzioni Go **restituiscono oltre al risultato dell'operazione anche un valore che descrive un eventuale errore** (del tipo `risultato, erro := funzioneChePuòFallire()`).  
Il chiamante poi controlla `err`, e se `err == nil`  --> nessun errore, altrimenti bisogna gestire l'errore.

Altra funzione importante che vedremo è `defer`, che permette di posticipare l'esecuzione di una funzione fino al termine della funzione corrente.  
Utilissima ad esempio per chiudere file o connessioni di rete: di solito ci si dimentica, ma con `defer` possiamo scriverlo subito dopo l'apertura del file/connessione e siamo sicuri che verrà eseguito solo alla fine della funzione, anche in caso di errori.

